# 금융위원회 기업기본정보: 계열회사 데이터 전처리

`금융위원회_기업기본정보_수집` 폴더의 `계열회사_페이지_1.csv`부터 `계열회사_페이지_4.csv`까지 병합하고, GraphRAG 구축에 필요한 아래 세 컬럼만 정제합니다.

- `crno`: 기준 기업 법인등록번호
- `afilCmpyNm`: 계열회사명
- `afilCmpyCrno`: 계열회사 법인등록번호

법인등록번호의 선행 0이 손실되지 않도록 모든 대상 컬럼을 문자열로 읽습니다. 공백·결측값·완전 중복 행을 제거하고, 두 등록번호가 13자리 숫자인지 검증한 뒤 UTF-8 BOM CSV로 저장합니다.

In [1]:
from pathlib import Path
from typing import Sequence

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    # 일반 Python 환경에서는 print를 표시 함수로 사용합니다.
    display = print

# 전처리 결과에 남길 컬럼 순서를 명시적으로 고정합니다.
KEEP_COLUMNS = ["crno", "afilCmpyNm", "afilCmpyCrno"]
DATA_DIR_NAME = "금융위원회_기업기본정보_수집"
INPUT_PATTERN = "계열회사_페이지_*.csv"
OUTPUT_FILE_NAME = "계열회사_전처리.csv"

## 1. 입력 경로와 파일 확인

노트북을 프로젝트 루트 또는 `notebooks` 폴더에서 실행해도 데이터 폴더를 찾도록 구성합니다. 입력 파일은 이름순으로 정렬하고 정확히 4개인지 확인합니다.

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    """데이터 폴더를 기준으로 프로젝트 루트 경로를 찾습니다.

    Args:
        start: 탐색을 시작할 경로. 기본값은 현재 작업 디렉터리입니다.

    Returns:
        금융위원회 데이터 폴더를 포함하는 프로젝트 루트 경로입니다.

    Raises:
        FileNotFoundError: 현재 경로와 상위 경로에서 데이터 폴더를 찾지 못한 경우.
    """
    start_path = (start or Path.cwd()).resolve()

    # 현재 위치부터 모든 상위 디렉터리를 차례대로 확인합니다.
    for candidate in (start_path, *start_path.parents):
        if (candidate / DATA_DIR_NAME).is_dir():
            return candidate

    raise FileNotFoundError(
        f"'{DATA_DIR_NAME}' 폴더를 찾을 수 없습니다. 프로젝트 내부에서 노트북을 실행하세요."
    )


def find_input_files(data_dir: Path, expected_count: int = 4) -> list[Path]:
    """계열회사 페이지 CSV 파일을 이름순으로 찾아 반환합니다.

    Args:
        data_dir: 원본 CSV 파일이 들어 있는 디렉터리입니다.
        expected_count: 반드시 존재해야 하는 입력 파일 개수입니다.

    Returns:
        파일명 기준으로 정렬된 CSV 경로 목록입니다.

    Raises:
        FileNotFoundError: 발견한 파일 개수가 기대값과 다른 경우.
    """
    input_files = sorted(data_dir.glob(INPUT_PATTERN))
    if len(input_files) != expected_count:
        raise FileNotFoundError(
            f"입력 파일은 {expected_count}개여야 하지만 {len(input_files)}개를 찾았습니다: "
            f"{[path.name for path in input_files]}"
        )
    return input_files

## 2. 병합·정제·검증 함수

입력 단계부터 필요한 컬럼만 문자열로 읽습니다. 정제 과정은 원본 DataFrame을 변경하지 않고 새 DataFrame을 반환합니다.

In [3]:
def load_and_merge_affiliates(csv_files: Sequence[Path]) -> pd.DataFrame:
    """여러 계열회사 CSV에서 필요한 세 컬럼만 읽어 하나로 병합합니다.

    법인등록번호가 숫자로 변환되어 선행 0이 사라지지 않도록 문자열 자료형을
    사용합니다. UTF-8 BOM 포함 여부와 관계없이 읽을 수 있도록 utf-8-sig를 사용합니다.

    Args:
        csv_files: 병합할 CSV 파일 경로 목록입니다.

    Returns:
        KEEP_COLUMNS 순서로 병합된 DataFrame입니다.

    Raises:
        ValueError: 입력 파일 목록이 비어 있는 경우.
        pandas.errors.ParserError: CSV 형식이 올바르지 않은 경우.
    """
    if not csv_files:
        raise ValueError("병합할 CSV 파일이 없습니다.")

    frames = []
    for csv_path in csv_files:
        frame = pd.read_csv(
            csv_path,
            usecols=KEEP_COLUMNS,
            dtype={column: "string" for column in KEEP_COLUMNS},
            encoding="utf-8-sig",
        )
        frames.append(frame)

    return pd.concat(frames, ignore_index=True)[KEEP_COLUMNS]


def clean_affiliates(data: pd.DataFrame) -> pd.DataFrame:
    """계열회사 데이터의 공백, 결측값, 완전 중복 행을 정리합니다.

    Args:
        data: KEEP_COLUMNS를 포함하는 원본 DataFrame입니다.

    Returns:
        세 컬럼만 남고 인덱스가 재설정된 정제 DataFrame입니다.

    Raises:
        KeyError: 필수 컬럼이 누락된 경우.
    """
    missing_columns = [column for column in KEEP_COLUMNS if column not in data.columns]
    if missing_columns:
        raise KeyError(f"필수 컬럼이 누락되었습니다: {missing_columns}")

    cleaned = data.loc[:, KEEP_COLUMNS].copy()
    for column in KEEP_COLUMNS:
        # 문자열 양끝의 불필요한 공백을 제거하고 빈 문자열은 결측값으로 통일합니다.
        cleaned[column] = cleaned[column].astype("string").str.strip().replace("", pd.NA)

    cleaned = cleaned.dropna(subset=KEEP_COLUMNS)
    cleaned = cleaned.drop_duplicates(subset=KEEP_COLUMNS, keep="first")
    return cleaned.reset_index(drop=True)


def validate_affiliates(data: pd.DataFrame) -> dict[str, int]:
    """정제 결과의 컬럼, 결측값, 중복, 법인등록번호 형식을 검증합니다.

    Args:
        data: 검증할 정제 DataFrame입니다.

    Returns:
        최종 행 수와 주요 오류 건수가 담긴 품질 요약 사전입니다.

    Raises:
        ValueError: 컬럼 순서가 다르거나 품질 기준을 만족하지 못한 경우.
    """
    if data.columns.tolist() != KEEP_COLUMNS:
        raise ValueError(
            f"결과 컬럼은 {KEEP_COLUMNS}이어야 합니다: {data.columns.tolist()}"
        )

    null_count = int(data[KEEP_COLUMNS].isna().sum().sum())
    duplicate_count = int(data.duplicated(subset=KEEP_COLUMNS).sum())
    invalid_crno_count = int((~data["crno"].str.fullmatch(r"\d{13}", na=False)).sum())
    invalid_afil_crno_count = int(
        (~data["afilCmpyCrno"].str.fullmatch(r"\d{13}", na=False)).sum()
    )

    summary = {
        "row_count": len(data),
        "null_count": null_count,
        "duplicate_count": duplicate_count,
        "invalid_crno_count": invalid_crno_count,
        "invalid_afil_cmpy_crno_count": invalid_afil_crno_count,
    }

    error_counts = {key: value for key, value in summary.items() if key != "row_count" and value}
    if error_counts:
        raise ValueError(f"데이터 품질 검증에 실패했습니다: {error_counts}")

    return summary

## 3. 함수 동작 확인

작은 예제 데이터로 양끝 공백, 결측값, 중복 행이 의도대로 처리되는지 먼저 확인합니다.

In [4]:
sample_data = pd.DataFrame(
    {
        "crno": [" 1101110000086 ", "1101110000086", None],
        "afilCmpyNm": [" 테스트계열사 ", "테스트계열사", "결측기업"],
        "afilCmpyCrno": ["1101110014764", "1101110014764", ""],
        "불필요한컬럼": [1, 2, 3],
    }
)
sample_cleaned = clean_affiliates(sample_data)
sample_summary = validate_affiliates(sample_cleaned)

assert sample_cleaned.shape == (1, 3)
assert sample_cleaned.iloc[0].to_dict() == {
    "crno": "1101110000086",
    "afilCmpyNm": "테스트계열사",
    "afilCmpyCrno": "1101110014764",
}
assert sample_summary["duplicate_count"] == 0
print("함수 동작 확인 완료")

함수 동작 확인 완료


## 4. 전체 데이터 전처리 및 저장

4개 파일을 병합·정제한 뒤 품질 검증을 통과한 데이터만 저장합니다. `utf-8-sig`는 Excel에서도 한글을 안정적으로 열 수 있는 UTF-8 BOM 형식입니다.

In [5]:
project_root = find_project_root()
data_dir = project_root / DATA_DIR_NAME
input_files = find_input_files(data_dir)
output_path = data_dir / OUTPUT_FILE_NAME

print("입력 파일:")
for input_file in input_files:
    print(f"- {input_file.name}")

merged_data = load_and_merge_affiliates(input_files)
cleaned_data = clean_affiliates(merged_data)
quality_summary = validate_affiliates(cleaned_data)

# 검증이 끝난 데이터만 UTF-8 BOM CSV로 저장합니다.
cleaned_data.to_csv(output_path, index=False, encoding="utf-8-sig")

processing_summary = pd.DataFrame(
    {
        "항목": ["입력 파일 수", "병합 행 수", "최종 행 수", "제거 행 수"],
        "값": [
            len(input_files),
            len(merged_data),
            len(cleaned_data),
            len(merged_data) - len(cleaned_data),
        ],
    }
)
display(processing_summary)
display(pd.Series(quality_summary, name="값").to_frame())
display(cleaned_data.head())
print(f"저장 완료: {output_path}")

입력 파일:
- 계열회사_페이지_1.csv
- 계열회사_페이지_2.csv
- 계열회사_페이지_3.csv
- 계열회사_페이지_4.csv


,항목,값
0,입력 파일 수,4
1,병합 행 수,40000
2,최종 행 수,9601
3,제거 행 수,30399


,값
row_count,9601
null_count,0
duplicate_count,0
invalid_crno_count,0
invalid_afil_cmpy_crno_count,0


,crno,afilCmpyNm,afilCmpyCrno
0,1101110000086,롯데건설(주),1101110014764
1,1101110000086,(주)롯데푸드,1101110033722
2,1101110000086,롯데지주(주),1101110076300
3,1101110000086,호텔롯데(주),1101110145410
4,1101110000086,롯데상사(주),1101110159099


저장 완료: C:\Users\Playdata\Desktop\김동석\교과목-2\mle-01-p2-team3\금융위원회_기업기본정보_수집\계열회사_전처리.csv
